# Data Ingestion & Engineering

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.sql.functions import col, when, isnan, count
import pandas as pd

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
spark = SparkSession.builder \
    .appName("HIGGS_BigData_Project") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

In [ ]:
# Schema Definition
columns = [
    "label",

    "lepton_pt",
    "lepton_eta",
    "lepton_phi",
    "missing_energy_magnitude",
    "missing_energy_phi",

    "jet1_pt",
    "jet1_eta",
    "jet1_phi",
    "jet1_btag",

    "jet2_pt",
    "jet2_eta",
    "jet2_phi",
    "jet2_btag",

    "jet3_pt",
    "jet3_eta",
    "jet3_phi",
    "jet3_btag",

    "jet4_pt",
    "jet4_eta",
    "jet4_phi",
    "jet4_btag",

    "m_jj",
    "m_jjj",
    "m_lv",
    "m_jlv",
    "m_bb",
    "m_wbb",
    "m_wwbb"
]

In [ ]:
schema = StructType([StructField(c, DoubleType(), True) for c in columns])

In [ ]:
# Data Ingestion
df = spark.read.csv(
    "/content/drive/MyDrive/HIGGS.csv.gz",
    schema=schema,
    header=False
)

In [ ]:
df.printSchema()

root
 |-- label: double (nullable = true)
 |-- lepton_pt: double (nullable = true)
 |-- lepton_eta: double (nullable = true)
 |-- lepton_phi: double (nullable = true)
 |-- missing_energy_magnitude: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- jet1_pt: double (nullable = true)
 |-- jet1_eta: double (nullable = true)
 |-- jet1_phi: double (nullable = true)
 |-- jet1_btag: double (nullable = true)
 |-- jet2_pt: double (nullable = true)
 |-- jet2_eta: double (nullable = true)
 |-- jet2_phi: double (nullable = true)
 |-- jet2_btag: double (nullable = true)
 |-- jet3_pt: double (nullable = true)
 |-- jet3_eta: double (nullable = true)
 |-- jet3_phi: double (nullable = true)
 |-- jet3_btag: double (nullable = true)
 |-- jet4_pt: double (nullable = true)
 |-- jet4_eta: double (nullable = true)
 |-- jet4_phi: double (nullable = true)
 |-- jet4_btag: double (nullable = true)
 |-- m_jj: double (nullable = true)
 |-- m_jjj: double (nullable = true)
 |-- m_lv: dou

In [ ]:
df.show(5)

+-----+------------------+-------------------+-------------------+------------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|label|         lepton_pt|         lepton_eta|         lepton_phi|missing_energy_magnitude| missing_energy_phi|           jet1_pt|            jet1_eta|           jet1_phi|         jet1_btag|           jet2_pt|           jet2_eta|           jet2_phi|         jet2_btag|           jet3_pt|            jet3_eta|           jet3_phi|        jet3_btag|            jet4_pt|            jet4_eta|            jet4_phi|        jet4_btag|              

In [ ]:
#  Data Validation
print("Total Records:", df.count())

Total Records: 11000000


In [ ]:
print("Total Columns:", len(df.columns))

Total Columns: 29


In [ ]:
# Missing Value Analysis
missing_df = df.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in df.columns
])

In [ ]:
missing_df.show()

+-----+---------+----------+----------+------------------------+------------------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+----+-----+----+-----+----+-----+------+
|label|lepton_pt|lepton_eta|lepton_phi|missing_energy_magnitude|missing_energy_phi|jet1_pt|jet1_eta|jet1_phi|jet1_btag|jet2_pt|jet2_eta|jet2_phi|jet2_btag|jet3_pt|jet3_eta|jet3_phi|jet3_btag|jet4_pt|jet4_eta|jet4_phi|jet4_btag|m_jj|m_jjj|m_lv|m_jlv|m_bb|m_wbb|m_wwbb|
+-----+---------+----------+----------+------------------------+------------------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+----+-----+----+-----+----+-----+------+
|    0|        0|         0|         0|                       0|                 0|      0|       0|       0|        0|      0|       0|       0|        0|      0|       0|       0|        0|     

In [ ]:
# Partitioning Strategy
df = df.repartition(200)

In [ ]:
# Storage Optimization
df.write.mode("overwrite").parquet("data/higgs_parquet")

In [ ]:
df = spark.read.parquet("data/higgs_parquet")

In [ ]:
df.cache()

DataFrame[label: double, lepton_pt: double, lepton_eta: double, lepton_phi: double, missing_energy_magnitude: double, missing_energy_phi: double, jet1_pt: double, jet1_eta: double, jet1_phi: double, jet1_btag: double, jet2_pt: double, jet2_eta: double, jet2_phi: double, jet2_btag: double, jet3_pt: double, jet3_eta: double, jet3_phi: double, jet3_btag: double, jet4_pt: double, jet4_eta: double, jet4_phi: double, jet4_btag: double, m_jj: double, m_jjj: double, m_lv: double, m_jlv: double, m_bb: double, m_wbb: double, m_wwbb: double]